In [1]:
!pip install pandas geopandas rasterio shapely pyproj openpyxl

   ---------------------------------------- 0.0/25.7 MB ? eta -:--:--
   ---------------------------------------- 0.1/25.7 MB 3.3 MB/s eta 0:00:08
   ---------------------------------------- 0.2/25.7 MB 3.6 MB/s eta 0:00:08
    --------------------------------------- 0.6/25.7 MB 4.8 MB/s eta 0:00:06
   - -------------------------------------- 0.8/25.7 MB 4.7 MB/s eta 0:00:06
   -- ------------------------------------- 1.4/25.7 MB 6.8 MB/s eta 0:00:04
   -- ------------------------------------- 1.9/25.7 MB 7.5 MB/s eta 0:00:04
   --- ------------------------------------ 2.0/25.7 MB 7.2 MB/s eta 0:00:04
   --- ------------------------------------ 2.6/25.7 MB 7.5 MB/s eta 0:00:04
   ---- ----------------------------------- 2.8/25.7 MB 7.7 MB/s eta 0:00:03
   ---- ----------------------------------- 3.0/25.7 MB 7.0 MB/s eta 0:00:04
   ----- ---------------------------------- 3.3/25.7 MB 7.0 MB/s eta 0:00:04
   ----- ---------------------------------- 3.7/25.7 MB 6.9 MB/s eta 0:00:04
   ---


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import geopandas as gpd
import pandas as pd

file_path = r"D:\Risk Scoring Palm Oil\data-samples.xlsx"
df = pd.read_excel(file_path)

# buat point geometry (lat/lon)
gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df["estate_lon"], df["estate_lat"]),
    crs="EPSG:4326"
)

# ubah ke CRS meter untuk buffer
gdf_meter = gdf.to_crs(3857)

# buffer 5 km
gdf_meter["geometry"] = gdf_meter.geometry.buffer(5000)

# convert kembali ke lat/lon
gdf_buffer = gdf_meter.to_crs(4326)

In [5]:
geom = gdf_buffer.iloc[0].geometry
print("Estate buffer bounds:", geom.bounds)

Estate buffer bounds: (105.36148623579402, -4.107803630563947, 105.45131776420597, -4.0181978809091765)


In [6]:
import glob

# tif_files = glob.glob(r"D:\Hansen Dataset\*.tif")

# tif_files

lossyear_files = glob.glob(r"D:\Hansen Dataset\lossyear\*.tif")

treecover_files = glob.glob(r"D:\Hansen Dataset\treecover\*.tif")

# penting digunakan agar pasangan tile sama
lossyear_files.sort()
treecover_files.sort()

In [7]:
from shapely.geometry import box
import rasterio

# for tif in tif_files:
#     with rasterio.open(tif) as src:
#         print(tif, geom.intersects(box(*src.bounds)))

for tif in lossyear_files:
    with rasterio.open(tif) as src:
        print(tif, geom.intersects(box(*src.bounds)))

D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_00N_090E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_00N_100E.tif True
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_00N_110E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_00N_120E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_00N_130E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_00N_140E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10N_090E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10N_100E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10N_110E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10N_120E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10N_130E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10S_120E.tif False
D:\Hansen Dataset\lossyear\Hansen_GFC-2024-v1.12_lossyear_10S_130E.tif False


In [8]:
#Fungsi menghitung forest loss
import numpy as np
import rasterio
from rasterio.mask import mask
from shapely.geometry import mapping, box

# def count_forest_loss(geometry, tif_list):

#     total_loss = 0

#     for tif in tif_list:

#         with rasterio.open(tif) as src:

#             raster_bbox = box(*src.bounds)

#             # skip raster yang tidak overlap
#             if not geometry.intersects(raster_bbox):
#                 continue

#             geom = [mapping(geometry)]

#             out_image, _ = mask(src, geom, crop=True)

#             data = out_image[0]

#             # buang nodata
#             data = data[data != src.nodata]

#             # hitung pixel forest loss
#             # total_loss += np.sum(data > 0)
#             total_loss += np.sum((data >= 1) & (data <= 24))

#     return int(total_loss)

def count_forest_loss(geometry, loss_files, treecover_files):

    total_loss = 0

    for loss_tif, tree_tif in zip(loss_files, treecover_files):

        with rasterio.open(loss_tif) as loss_src, rasterio.open(tree_tif) as tree_src:

            raster_bbox = box(*loss_src.bounds)

            if not geometry.intersects(raster_bbox):
                continue

            geom = [mapping(geometry)]

            loss_img, _ = mask(loss_src, geom, crop=True)
            tree_img, _ = mask(tree_src, geom, crop=True)

            loss = loss_img[0]
            tree = tree_img[0]

            # forest definition
            condition = (tree >= 30) & (loss >= 1)

            total_loss += np.sum(condition)

    return int(total_loss)

In [9]:
# #Hitung forest loss untuk semua estate
# gdf_buffer["forest_loss_pixels"] = gdf_buffer.geometry.apply(
#     lambda geom: count_forest_loss(geom, tif_files)
# )

# Hitung forest loss untuk semua estate
gdf_buffer["forest_loss_pixels"] = gdf_buffer.geometry.apply(
    lambda geom: count_forest_loss(geom, lossyear_files, treecover_files)
)

In [11]:
# gdf_buffer[["estate_id","forest_loss_pixels"]].head()

In [12]:
gdf_buffer = gdf_buffer.drop(
    columns=["forest_loss_pixels","forest_loss_ha"],
    errors="ignore"
)

In [13]:
estate_unique = gdf_buffer.drop_duplicates(subset="estate_id").copy()

# estate_unique["forest_loss_pixels"] = estate_unique.geometry.apply(
#     lambda geom: count_forest_loss(geom, tif_files)
# )

estate_unique["forest_loss_pixels"] = estate_unique.geometry.apply(
    lambda geom: count_forest_loss(geom, lossyear_files, treecover_files)
)

estate_unique["forest_loss_ha"] = estate_unique["forest_loss_pixels"] * 0.0956

gdf_final = gdf_buffer.merge(
    estate_unique[["estate_id","forest_loss_pixels","forest_loss_ha"]],
    on="estate_id",
    how="left"
)

In [14]:
gdf_final["loss_density"] = (
    gdf_final["forest_loss_ha"] /
    gdf_final["luas_area_supply_base (hectare)"]
)

In [16]:
gdf_final[[
    # "estate_id",
    "luas_area_supply_base (hectare)",
    "forest_loss_ha",
    "loss_density"
]].head()

,luas_area_supply_base (hectare),forest_loss_ha,loss_density
0,2.0,3398.9624,1699.4812
1,4.0,392.3424,98.0856
2,NaN,3398.9624,NaN
3,4.0,392.3424,98.0856
4,1.0,1422.8148,1422.8148


In [17]:
import geopandas as gpd
from shapely.geometry import Point

gdf["mill_geometry"] = gdf.apply(
    lambda row: Point(row["mill_lon"], row["mill_lat"]),
    axis=1
)

gdf_mill = gpd.GeoDataFrame(
    gdf,
    geometry="mill_geometry",
    crs="EPSG:4326"
)

In [18]:
mills = gdf_mill[["mill_id", "mill_geometry"]].drop_duplicates()

In [19]:
#ubah ke CRS meter agar buffer akurat
mills = mills.to_crs("EPSG:3857")

#buat buffer 50 km
mills["buffer_50km"] = mills.geometry.buffer(50000)

#jadikan buffer sebagai geometry
mills = mills.set_geometry("buffer_50km")

#kembalikan CRS ke lat/lon agar cocok dengan raster Hansen
mills = mills.to_crs("EPSG:4326")

In [20]:
from rasterio.mask import mask
from shapely.geometry import box, mapping
import rasterio
import numpy as np

# results = []

# # pastikan mill unik
# mills_unique = mills.drop_duplicates(subset="mill_id")

# for idx, row in mills_unique.iterrows():

#     geom = row["buffer_50km"]
#     total_pixels = 0

#     for tif in tif_files:

#         with rasterio.open(tif) as src:

#             # cek apakah buffer mill overlap tile
#             if geom.intersects(box(*src.bounds)):

#                 out_image, _ = mask(
#                     src,
#                     [mapping(geom)],
#                     crop=True,
#                     filled=False   # penting supaya nodata tidak dihitung
#                 )

#                 data = out_image[0]

#                 # hitung hanya pixel forest loss (2001–2024)
#                 loss_pixels = np.sum((data >= 1) & (data <= 24))

#                 total_pixels += loss_pixels

#     # konversi pixel → hektar
#     loss_ha = total_pixels * 0.0956

#     results.append({
#         "mill_id": row["mill_id"],
#         "forest_loss_50km_ha": loss_ha
#     })

results = []

mills_unique = mills.drop_duplicates(subset="mill_id")

for idx, row in mills_unique.iterrows():

    geom = row["buffer_50km"]
    total_pixels = 0

    for loss_tif, tree_tif in zip(lossyear_files, treecover_files):

        with rasterio.open(loss_tif) as loss_src, rasterio.open(tree_tif) as tree_src:

            if geom.intersects(box(*loss_src.bounds)):

                loss_img, _ = mask(
                    loss_src,
                    [mapping(geom)],
                    crop=True,
                    filled=False
                )

                tree_img, _ = mask(
                    tree_src,
                    [mapping(geom)],
                    crop=True,
                    filled=False
                )

                loss = loss_img[0]
                tree = tree_img[0]

                condition = (tree >= 30) & (loss >= 1)

                total_pixels += np.sum(condition)

    loss_ha = total_pixels * 0.0956

    results.append({
        "mill_id": row["mill_id"],
        "forest_loss_50km_ha": loss_ha
    })

In [21]:
import pandas as pd

mill_loss = pd.DataFrame(results)
# mill_loss

In [20]:
gdf_final.columns

Index(['entity_id', 'vendor_id', 'mill_id', 'estate_id', 'estate_lat',
       'estate_lon', 'mill_lat', 'mill_lon', 'distance_km',
       'luas_area_supply_base (hectare)', 'geometry', 'forest_loss_pixels_x',
       'forest_loss_pixels_y', 'forest_loss_ha', 'loss_density'],
      dtype='object')

In [22]:
gdf_final = gdf_final.drop(columns=["forest_loss_50km_ha"], errors="ignore")

gdf_final = gdf_final.merge(
    mill_loss[["mill_id", "forest_loss_50km_ha"]],
    on="mill_id",
    how="left"
)

# gdf_final.head()

In [25]:
BUFFER_AREA_50KM_HA = 785398

gdf_final["mill_loss_density"] = (
    gdf_final["forest_loss_50km_ha"] /
    BUFFER_AREA_50KM_HA
)

In [2]:
# gdf_final[[
#     "mill_id",
#     "forest_loss_50km_ha",
#     "mill_loss_density"
# ]].head()

In [29]:
def count_forest2000(geom, treecover_files):

    total_pixels = 0

    for tif in treecover_files:

        with rasterio.open(tif) as src:

            if geom.intersects(box(*src.bounds)):

                out_image, _ = mask(
                    src,
                    [mapping(geom)],
                    crop=True,
                    filled=False
                )

                data = out_image[0]

                forest_pixels = np.sum(data >= 30)

                total_pixels += forest_pixels

    return total_pixels

In [31]:
mills_unique = mills.drop_duplicates(subset="mill_id").copy()

In [32]:
mills_unique["forest2000_pixels"] = mills_unique.geometry.apply(
    lambda geom: count_forest2000(geom, treecover_files)
)

mills_unique["forest2000_ha"] = mills_unique["forest2000_pixels"] * 0.09

In [34]:
gdf_final = gdf_final.merge(
    mills_unique[["mill_id", "forest2000_ha"]],
    on="mill_id",
    how="left"
)

In [37]:
gdf_final["deforestation_rate"] = (
    gdf_final["forest_loss_50km_ha"] /
    gdf_final["forest2000_ha"].replace(0, np.nan)
)

In [1]:
# gdf_final[[
#     "mill_id",
#     "forest_loss_50km_ha",
#     "forest2000_ha",
#     "deforestation_rate"
# ]].head()

In [40]:
gdf_final.to_excel("gdf_final.xlsx", index=False)